# Test2

The test of our proposed algorithm.

## Implementation

copied from the networkx/drawing/layout.py

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import time
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx


def make_fig(
    Gs: list,
    methods: list,
    file_name: str,
    legend_pos=1.0,
    rect_pos=0.96,
    rows=2,
    fontsize=30,
):
    assert len(Gs) % rows == 0

    n2 = len(Gs) // rows
    fig, axes = plt.subplots(
        rows * len(methods), n2, figsize=(3 * n2, 3 * rows * len(methods))
    )
    axes = axes.flatten()

    for i, (G, graph_name) in enumerate(Gs):
        for j, (draw_func, _, node_color, kwargs) in enumerate(methods):
            ax = axes[i % n2 + j * n2 + len(methods) * n2 * int(i / n2)]

            t0 = time.perf_counter()
            try:
                pos = draw_func(G, **kwargs)
            except ValueError as e:
                print(f"Error: {e}")
                # kamada_kawai_layout does not support negative edge weights
                pos = np.zeros((len(G), 2))
            t1 = time.perf_counter()
            print(f"{draw_func.__name__} took {t1 - t0:.2f}s")

            if type(pos) == np.ndarray:
                nodes = G.nodes()
                pos = dict(zip(nodes, pos))

            if j == 0:
                if graph_name.endswith("_graph"):
                    graph_name = graph_name[:-6]
                if graph_name == "dorogovtsev_goltsev_mendes":
                    graph_name = "DGM"
                ax.set_title(f"{graph_name}\n{t1 - t0:.2f}s", fontsize=20)
            else:
                ax.set_title(f"{t1 - t0:.2f}s", fontsize=20)

            # cmap = plt.get_cmap("jet")
            # n = G.number_of_nodes()
            # colorMap = np.array([cmap(i / (n - 1)) for i in range(n)])

            nx.draw(
                G,
                pos=pos,
                ax=ax,
                node_size=min(50, max(1, 5000 // len(G))),
                node_color=node_color,
                # node_color=colorMap,
            )
            ax.axis("on")

    handles = []
    for _, func_name, color, _ in methods:
        handles.append(mpatches.Patch(color=color, label=func_name))

    fig.legend(
        handles=handles,
        loc="upper center",
        ncol=len(handles),
        bbox_to_anchor=(0.5, legend_pos),
        fontsize=fontsize,
    )

    plt.tight_layout(rect=(0.0, 0.0, 1.0, rect_pos))
    plt.savefig(file_name, dpi=300)
    plt.close()

This is the utility function for the test.

## Comparison

### NetworkX Graphs

In [2]:
import networkx as nx
import scipy.sparse
import ssgetpy
import scipy


def graph_generator(n: int):
    matrixes = [ssgetpy.search(name) for name in ["jagmesh1", "dwt_2680"]]

    Gs = []
    for mat in matrixes:
        mat = mat[0]
        path = mat.download(extract=True)[0]
        A = scipy.io.mmread(path + f"/{mat.name}.mtx")
        G = nx.from_scipy_sparse_array(A)
        G.remove_edges_from(nx.selfloop_edges(G))
        Gs.append((G, mat.name))
        print(f"{mat.name}: {len(G)} nodes, {len(G.edges)} edges")

    Gs.extend(
        [
            (nx.complete_graph(n), "complete"),
            (nx.cycle_graph(n), "cycle"),
        ]
    )

    return Gs


def make_methods(iterations: int):
    return [
        (
            nx.spring_layout,
            "old version (force)",
            "tab:blue",
            {"iterations": iterations, "method": "force", "seed": 0},
        ),
        (
            nx.spring_layout,
            "new version (energy)",
            "tab:orange",
            {"iterations": iterations, "method": "energy", "seed": 0},
        ),
    ]


make_fig(graph_generator(500), make_methods(150), "tsukuba1", 1.03, 0.88, 1)

/home/hari64boli64/University/networkx/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


jagmesh1: 936 nodes, 2664 edges
dwt_2680: 2680 nodes, 11173 edges
spring_layout took 12.75s
spring_layout took 14.80s
spring_layout took 71.29s
spring_layout took 63.96s
spring_layout took 13.61s
spring_layout took 6.51s
spring_layout took 7.99s
spring_layout took 6.42s


<!-- ### Unconnected Graphs -->

In [3]:
def graph_generator2():
    Gs = []

    n = 3
    G1 = nx.grid_2d_graph(n, n)
    G2 = nx.cycle_graph(n * n)
    G = nx.disjoint_union(G1, G2)

    Gs.append((G.copy(), "union"))

    G = G.copy()
    v = G.number_of_nodes()
    G.add_edge(v, 0, weight=-1)
    G.add_edge(v, n**2, weight=-1)
    Gs.append((G.copy(), "negative"))

    G = nx.erdos_renyi_graph(500, 0)
    Gs.append((G, "no edges"))

    return Gs


make_fig(graph_generator2(), make_methods(300), "tsukuba2", 1.00, 0.88, 1, 20)

spring_layout took 0.02s
spring_layout took 0.11s
spring_layout took 0.03s
spring_layout took 0.10s
spring_layout took 11.70s
spring_layout took 2.88s
